# Simple cross-asset momentum

Monthly-rebalanced, equal-weight long on the top-2 of {SPY, IEF, GLD, DBC, VNQ} by 12-1 month return.

Notebook stays thin: all reusable logic lives in `alpha_lab.data` / `alpha_lab.backtest` / `alpha_lab.reporting`.

In [ ]:
import pandas as pd

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly

UNIVERSE = ["SPY", "IEF", "GLD", "DBC", "VNQ"]
START = "2010-01-01"
END = None

In [ ]:
prices = load_prices(UNIVERSE, start=START, end=END)
prices.tail()

In [ ]:
# 12-1 month momentum: return over the past 12 months excluding the most recent month.
monthly = prices.resample("ME").last()
mom = monthly.shift(1) / monthly.shift(12) - 1

# Equal-weight long top-2 each month.
top_k = 2
ranks = mom.rank(axis=1, ascending=False)
weights_monthly = (ranks <= top_k).astype(float) / top_k
weights_monthly = weights_monthly.where(mom.notna(), 0.0)

# Expand to daily on the prices calendar (run_backtest will pick rebalance dates).
weights = weights_monthly.reindex(prices.index).ffill().fillna(0.0)
weights.tail()

In [ ]:
res = run_backtest(weights, prices, rebalance="ME", costs_bps=1.0, slippage_bps=2.0)
pd.Series(summary(res.returns))

In [ ]:
equity_curve(res.returns, name="top-2 momentum")

In [ ]:
drawdown_chart(res.returns)

In [ ]:
heatmap_monthly(res.returns)